In [1]:
import networkx as nx
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd

In [73]:
## Import data from text file and import list of "essential" proteins

raw = nx.read_weighted_edgelist("protein.links.clean.txt",comments="#",nodetype=str)
raw_phys = nx.read_weighted_edgelist("protein.physical.links.clean.txt",comments="#",nodetype=str)

essential = pd.read_csv("Data - List of essential proteins of saccharomyces cerevisiae.csv", header=None)
ess_prot = set(essential.iloc[:,1])

ess_graph = raw.subgraph(ess_prot).copy()
## ----------------------------------------------------------------

In [76]:
target = "YMR161W"
confidence = 750  ## Minimum edge strength (anything below ignored)

graph = raw.copy()
physical = raw_phys.copy()

# removes any edges below the confidence score
for e in graph.edges:
    weight = list(graph.get_edge_data(e[0],e[1]).values())
    if(weight[0] <= confidence):
        graph.remove_edge(e[0],e[1])

# Identifies largest connected component and ignores any disconnected nodes
largest = max(nx.connected_components(graph), key=len)
graph = graph.subgraph(largest).copy()


## Create a copy with the target node removed for analysis
if target in graph:
    edit = graph.copy()
    edit.remove_node(target)
else:
    print(f"{target} not included in network") 


# Repeat for Physical links data -------------------------------------------------------------------
# --------------------------------------------------------------------------------------------------

# removes any edges below the confidence score
for e in physical.edges:
    weight = list(physical.get_edge_data(e[0],e[1]).values())
    if(weight[0] <= confidence):
        physical.remove_edge(e[0],e[1])

# Identifies largest connected component and ignores any disconnected nodes
largest = max(nx.connected_components(physical), key=len)
physical = physical.subgraph(largest).copy()

## Create a copy with the target node removed for analysis
if target in physical:
    phys_edit = physical.copy()
    phys_edit.remove_node(target)
else:
    print(f"{target} not included in physical network") 

YMR161W not included in physical network


In [89]:
## Create set of nodes local to target node

# note distance of 4: 4586, 3: 2437, 2: 254, ___ (5519 total)
distance = 1

local_nodes = set(nx.single_source_shortest_path_length(graph, target, cutoff=distance).keys())


print(len(local_nodes))
print(len(graph))

20
5519


In [69]:
print(nx.shortest_path(graph, "YMR161W", "YJL162C"))
print(local_nodes)

['YMR161W', 'YFR041C', 'YJL162C']
{'YBL075C', 'YNL328C', 'YHR064C', 'YLR008C', 'YGL128C', 'YMR186W', 'YLL024C', 'YMR161W', 'YEL030W', 'YAL005C', 'YER103W', 'YPR061C', 'YGR285C', 'YJL034W', 'YJR097W', 'YKL073W', 'YFR041C', 'YGL018C', 'YDR320C', 'YPL106C'}


In [79]:
# Compare with and without HLJ1

control = nx.betweenness_centrality(graph)
measure = nx.betweenness_centrality(edit)



In [92]:
difference = {node: measure[node] - control[node] for node in edit.nodes}

relative = {
    node: difference[node] / measure[node]
    for node in edit.nodes
    if measure[node] != 0
}

top_20_rel = sorted(relative.items(), key=lambda x: -x[1])[:20]

not_local_rel = [(node, diff) for node, diff in top_20_rel if node not in local_nodes]

for node, diff in not_local_rel:
    print(f"{node}: {diff:+.5f}")

for node in top_20_rel:
    print(node)



YJL162C: +0.03075
YNL077W: +0.01821
YLR090W: +0.01659
YHR110W: +0.01085
YFL016C: +0.00737
YER048C: +0.00257
YNL281W: +0.00221
YIR004W: +0.00156
YOR027W: +0.00134
YMR214W: +0.00112
YMR147W: +0.00103
YPR037C: +0.00091
YDR214W: +0.00088
YNL007C: +0.00087
YNL066W: +0.00081
YDL018C: +0.00080
YAR002C-A: +0.00077
YKL117W: +0.00076
YBR169C: +0.00075
YHR178W: +0.00071
('YJL162C', 0.030752931336883945)
('YNL077W', 0.018211355627661397)
('YLR090W', 0.016593049635308894)
('YHR110W', 0.010849787909749201)
('YFL016C', 0.007365189517118603)
('YER048C', 0.0025708495326068432)
('YNL281W', 0.0022068202906215134)
('YIR004W', 0.001562295788898712)
('YOR027W', 0.0013388643668801141)
('YMR214W', 0.0011231218190898245)
('YMR147W', 0.0010274016804817858)
('YPR037C', 0.0009091457252805499)
('YDR214W', 0.0008833032317302996)
('YNL007C', 0.0008736032113530658)
('YNL066W', 0.00080703423303224)
('YDL018C', 0.0008025762679995926)
('YAR002C-A', 0.000765574931198229)
('YKL117W', 0.0007561765460875019)
('YBR169C', 0.0

In [93]:
top_20_abs = sorted(difference.items(), key=lambda x: -x[1])[:20]

not_local_abs = [(node, diff) for node, diff in top_20_abs if node not in local_nodes]

for node, diff in not_local_abs:
    print(f"{node}: {diff:+.5f}")

for node in top_20_abs:
    print(node)

YLR167W: +0.00001
YBR010W: +0.00001
YFL038C: +0.00001
YPL240C: +0.00001
YNL178W: +0.00001
YKL192C: +0.00000
YFL039C: +0.00000
YLL013C: +0.00000
YER133W: +0.00000
YMR014W: +0.00000
YEL021W: +0.00000
YIL148W: +0.00000
YER125W: +0.00000
YKR094C: +0.00000
YMR217W: +0.00000
YJL189W: +0.00000
YDL140C: +0.00000
YAL038W: +0.00000
YEL009C: +0.00000
YER095W: +0.00000
('YLR167W', 8.345977634174884e-06)
('YBR010W', 6.9670096261229e-06)
('YFL038C', 6.145812850501253e-06)
('YPL240C', 5.9908066184603814e-06)
('YNL178W', 5.233851751331245e-06)
('YKL192C', 4.229746538729831e-06)
('YFL039C', 4.034001192830539e-06)
('YLL013C', 3.701390795980697e-06)
('YER133W', 3.573735324285607e-06)
('YMR014W', 3.4934820988090365e-06)
('YEL021W', 3.3770579077156693e-06)
('YIL148W', 3.2358679767242188e-06)
('YER125W', 3.210280688807285e-06)
('YKR094C', 3.0768687342350165e-06)
('YMR217W', 3.0699814310471835e-06)
('YJL189W', 3.038550875971921e-06)
('YDL140C', 2.9179404663452757e-06)
('YAL038W', 2.8511615431745657e-06)
('YE

In [96]:
print(relative["YJL034W"])

-0.007691454911125111
